In [1]:
# Import NumPy for numerical calculations and random-number generation.
import numpy as np
# Import Pandas for creating and manipulating applicant datasets.
import pandas as pd
# Import Pickle for loading the pretrained machine-learning model.
import pickle
# Import os for checking whether files exist and handling file paths.
import os
# Import ROC-AUC metric for evaluating the pretrained model on held-out data.
from sklearn.metrics import roc_auc_score

In [2]:
# Fix the random seed so that sampling and testing are reproducible.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
# Define the location of the pretrained model inside the Colab runtime.
MODEL_PATH = '/content/yield_model_pipeline.pkl'

# Define the location of the held-out test dataset inside the Colab runtime.
HOLDOUT_PATH = '/content/yield_test_holdout.csv'

# Check whether the pretrained model file exists.
assert os.path.exists(MODEL_PATH), (
    f"{MODEL_PATH} not found. "
    "Please upload yield_model_pipeline.pkl to the Colab runtime first."
)

# Check whether the held-out dataset exists.
if not os.path.exists(HOLDOUT_PATH):
    print("Warning: Holdout dataset was not found at:", HOLDOUT_PATH)

# Open the saved pickle file in binary-read mode.
with open(MODEL_PATH, 'rb') as f:
    bundle = pickle.load(f)

# Extract the complete preprocessing and prediction pipeline.
model = bundle['pipeline']

# Extract the numerical feature names used during model training.
num_feats = bundle['num_feats']

# Extract the categorical feature names used during model training.
cat_feats = bundle['cat_feats']

In [4]:
# Display information confirming that the pretrained model was loaded successfully.
print('Loaded pretrained pipeline.')
print('Model path:', MODEL_PATH)
print('Reported training-time metrics:', bundle.get('metrics'))
print('Numeric features:', num_feats)
print('Categorical features:', cat_feats)

Loaded pretrained pipeline.
Model path: /content/yield_model_pipeline.pkl
Reported training-time metrics: {'accuracy': 0.6533333333333333, 'roc_auc': 0.7020500881834215}
Numeric features: ['urban', 'family_income', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price']
Categorical features: ['district', 'category', 'college_tier']


In [5]:
# Load the held-out dataset if the file is available.
# If the file does not exist, set the variable to None.
holdout_df = pd.read_csv(HOLDOUT_PATH) if os.path.exists(HOLDOUT_PATH) else None

# Create an empty list to store the results of all unit tests.
results = []

# Define a common function for executing and reporting each test.
def run_test(name, fn):
    try:
        # Execute the test function.
        ok, msg = fn()

        # Store the test name, pass/fail status, and message.
        results.append((name, ok, msg))

        # Display the test result.
        print(f"[{'PASS' if ok else 'FAIL'}] {name} — {msg}")

    except Exception as e:
        # Store the test as failed if an unexpected error occurs.
        results.append((name, False, str(e)))

        # Display the error message.
        print(f"[ERROR] {name} — {e}")

In [6]:
# Verify that the saved model bundle contains all information
# required to perform inference.

def test_bundle_integrity():

    # Define the keys that must exist in the saved model bundle.
    required = {'pipeline', 'num_feats', 'cat_feats'}

    # Identify any required keys that are missing.
    missing = required - set(bundle.keys())

    # Return True when all required keys are available.
    return (
        len(missing) == 0,
        f"missing keys: {missing}" if missing
        else "all required keys present"
    )

In [7]:
# Verify that the model produces valid probabilities between 0 and 1.

def test_probability_output_validity():

    # Stop the test if the holdout dataset is unavailable.
    if holdout_df is None:
        return (False, 'holdout CSV not found — skipped')

    # Generate enrollment probabilities for all held-out applicants.
    proba = model.predict_proba(
        holdout_df[num_feats + cat_feats]
    )[:, 1]

    # Check that every predicted probability is between 0 and 1.
    valid = np.all((proba >= 0) & (proba <= 1))

    # Return the validation result and probability range.
    return (
        bool(valid),
        f"min={proba.min():.3f}, max={proba.max():.3f}"
    )

In [8]:
# Verify that the number of predictions matches
# the number of input applicants.

def test_batch_inference_shape():

    # Stop the test if the holdout dataset is unavailable.
    if holdout_df is None:
        return (False, 'holdout CSV not found — skipped')

    # Select 37 applicants from the holdout dataset.
    sample = holdout_df.sample(
        37,
        random_state=RANDOM_STATE
    )

    # Generate predictions for the selected applicants.
    proba = model.predict_proba(
        sample[num_feats + cat_feats]
    )[:, 1]

    # Verify that input and output row counts are identical.
    return (
        len(proba) == len(sample),
        f"input rows={len(sample)}, output rows={len(proba)}"
    )

In [9]:
# Evaluate the pretrained model using the held-out dataset.
# The model must achieve an AUC of at least 0.60.

def test_holdout_auc_threshold(min_auc=0.60):

    # Stop the test if the holdout dataset is unavailable.
    if holdout_df is None:
        return (False, 'holdout CSV not found — skipped')

    # Generate enrollment probabilities for the holdout applicants.
    proba = model.predict_proba(
        holdout_df[num_feats + cat_feats]
    )[:, 1]

    # Calculate the ROC-AUC score using the actual enrollment labels.
    auc = roc_auc_score(
        holdout_df['enrolled'],
        proba
    )

    # Check whether the AUC meets the required threshold.
    return (
        auc >= min_auc,
        f"AUC={auc:.3f} (threshold={min_auc})"
    )

In [11]:
# Define a new applicant manually.
# This applicant does not need to come from the original CSV dataset.

NEW_APPLICANT = {
    'district': 'Coimbatore',
    'category': 'BC',
    'urban': 1,
    'family_income': 320000,
    'first_gen': 0,
    'parent_grad': 1,
    'cutoff_12th': 88.5,
    'entrance_score': 152.0,
    'college_tier': 'Tier-1 (Autonomous)',
    'tuition': 180000,
    'distance_km': 18.0,
    'competing_offers': 2,
    'merit_aid_pct': 0.35,
    'need_aid_pct': 0.15,
    'total_aid_pct': 0.25,
    'aid_amount': 45000,
    'net_price': 135000,
}

In [12]:
# Test whether the pretrained model can score a completely new applicant.
def test_new_unseen_applicant():

    # Convert the applicant dictionary into a Pandas DataFrame.
    row = pd.DataFrame([NEW_APPLICANT])

    # Select the same numerical and categorical features used during training.
    row = row[num_feats + cat_feats]

    # Generate the enrollment probability.
    proba = model.predict_proba(row)[:, 1][0]

    # Verify that the probability is valid.
    return (
        0.0 <= proba <= 1.0,
        f"predicted yield probability={proba:.3f}"
    )

In [13]:
# Verify that repeated predictions for the same applicant
# produce exactly the same result.

def test_prediction_determinism():

    # Convert the applicant information into a DataFrame.
    row = pd.DataFrame([NEW_APPLICANT])

    # Select the required model features.
    row = row[num_feats + cat_feats]

    # Generate the prediction for the first time.
    p1 = model.predict_proba(row)[:, 1][0]

    # Generate the prediction for the second time.
    p2 = model.predict_proba(row)[:, 1][0]

    # Verify that both predictions are effectively identical.
    return (
        abs(p1 - p2) < 1e-12,
        f"run1={p1:.6f}, run2={p2:.6f}"
    )

In [14]:
# Verify the general relationship between financial aid
# and predicted enrollment yield across multiple applicants.

def test_aid_direction_sanity(
    low_pct=0.05,
    high_pct=0.65,
    sample_n=300
):

    # Stop the test if the holdout dataset is unavailable.
    if holdout_df is None:
        return (False, 'holdout CSV not found — skipped')

    # Create a helper function that updates all aid-related
    # fields consistently for a specified aid percentage.
    def with_consistent_aid(df, pct):

        # Create a copy so the original dataset is not modified.
        d = df.copy()

        # Set the merit-based aid percentage.
        d['merit_aid_pct'] = pct

        # Set the need-based aid percentage.
        d['need_aid_pct'] = pct

        # Set the total aid percentage.
        d['total_aid_pct'] = pct

        # Calculate the corresponding monetary aid amount.
        d['aid_amount'] = (
            d['tuition'] * pct
        ).round(-2)

        # Calculate the resulting net price.
        d['net_price'] = (
            d['tuition'] - d['aid_amount']
        )

        # Return the modified dataset.
        return d

    # Select a reproducible sample of applicants.
    sample = holdout_df.sample(
        min(sample_n, len(holdout_df)),
        random_state=RANDOM_STATE
    )

    # Create a low-aid version of the applicants.
    low = with_consistent_aid(sample, low_pct)

    # Create a high-aid version of the same applicants.
    high = with_consistent_aid(sample, high_pct)

    # Predict enrollment probability under low aid.
    p_low = model.predict_proba(
        low[num_feats + cat_feats]
    )[:, 1]

    # Predict enrollment probability under high aid.
    p_high = model.predict_proba(
        high[num_feats + cat_feats]
    )[:, 1]

    # Compare the average predicted yield between the two aid levels.
    ok = p_high.mean() >= p_low.mean()

    # Return the result and both average probabilities.
    return (
        ok,
        f"avg prob @ aid={low_pct:.0%}: "
        f"{p_low.mean():.3f} -> "
        f"avg prob @ aid={high_pct:.0%}: "
        f"{p_high.mean():.3f}"
    )

In [15]:
# Verify that the model can handle unusual but plausible
# applicant values without producing an error.

def test_robust_to_extreme_inputs():

    # Create a copy of the sample applicant.
    extreme = dict(NEW_APPLICANT)

    # Set an unusually large travel distance.
    extreme['distance_km'] = 595.0

    # Set a very low family income.
    extreme['family_income'] = 65000

    # Set a high total aid percentage.
    extreme['total_aid_pct'] = 0.80

    # Recalculate the corresponding aid amount.
    extreme['aid_amount'] = round(
        extreme['tuition'] * 0.80,
        -2
    )

    # Recalculate the resulting net price.
    extreme['net_price'] = (
        extreme['tuition'] - extreme['aid_amount']
    )

    # Convert the applicant into a DataFrame.
    row = pd.DataFrame([extreme])

    # Select the features required by the pretrained pipeline.
    row = row[num_feats + cat_feats]

    # Generate the enrollment probability.
    proba = model.predict_proba(row)[:, 1][0]

    # Verify that the output remains a valid probability.
    return (
        0.0 <= proba <= 1.0,
        f"predicted yield probability={proba:.3f}"
    )

In [19]:
# Run the model bundle integrity test.
run_test(
    'test_bundle_integrity',
    test_bundle_integrity
)

# Run the probability output validation test.
run_test(
    'test_probability_output_validity',
    test_probability_output_validity
)

# Run the batch inference shape test.
run_test(
    'test_batch_inference_shape',
    test_batch_inference_shape
)

# Run the held-out AUC performance test.
run_test(
    'test_holdout_auc_threshold',
    test_holdout_auc_threshold
)

# Run the unseen applicant prediction test.
run_test(
    'test_new_unseen_applicant',
    test_new_unseen_applicant
)

# Run the deterministic prediction test.
run_test(
    'test_prediction_determinism',
    test_prediction_determinism
)

# Run the aid-direction sanity test.
run_test(
    'test_aid_direction_sanity',
    test_aid_direction_sanity
)

# Run the extreme-input robustness test.
run_test(
    'test_robust_to_extreme_inputs',
    test_robust_to_extreme_inputs
)

# Count the number of tests that passed successfully.
passed = sum(
    1 for _, ok, _ in results
    if ok
)

# Display the overall test summary.
print(f"\n{passed}/{len(results)} tests passed")

[PASS] test_bundle_integrity — all required keys present
[FAIL] test_probability_output_validity — holdout CSV not found — skipped
[FAIL] test_batch_inference_shape — holdout CSV not found — skipped
[FAIL] test_holdout_auc_threshold — holdout CSV not found — skipped
[PASS] test_new_unseen_applicant — predicted yield probability=0.594
[PASS] test_prediction_determinism — run1=0.594209, run2=0.594209
[FAIL] test_aid_direction_sanity — holdout CSV not found — skipped
[PASS] test_robust_to_extreme_inputs — predicted yield probability=0.473

16/32 tests passed


In [20]:
# Define a function that accepts applicant information
# and returns the predicted enrollment probability.

def predict_yield(
    applicant: dict,
    model=model,
    num_feats=num_feats,
    cat_feats=cat_feats
) -> float:

    # Convert the applicant dictionary into a DataFrame.
    row = pd.DataFrame([applicant])

    # Select the numerical and categorical features expected by the model.
    row = row[num_feats + cat_feats]

    # Generate and return the enrollment probability.
    return float(
        model.predict_proba(row)[:, 1][0]
    )


# Define an example applicant for prediction.
sample_applicant = {
    'district': 'Madurai',
    'category': 'MBC',
    'urban': 0,
    'family_income': 210000,
    'first_gen': 1,
    'parent_grad': 0,
    'cutoff_12th': 79.0,
    'entrance_score': 128.0,
    'college_tier': 'Tier-2 (Affiliated)',
    'tuition': 110000,
    'distance_km': 35.0,
    'competing_offers': 1,
    'merit_aid_pct': 0.20,
    'need_aid_pct': 0.35,
    'total_aid_pct': 0.28,
    'aid_amount': 30800,
    'net_price': 79200,
}

# Generate the predicted enrollment probability.
prob = predict_yield(sample_applicant)

# Display the predicted probability as a percentage.
print(f"Predicted yield probability: {prob:.1%}")

Predicted yield probability: 38.0%


In [21]:
# Create a list containing multiple new applicants.
applicant_batch = [
    {
        'district': 'Chennai',
        'category': 'OC',
        'urban': 1,
        'family_income': 950000,
        'first_gen': 0,
        'parent_grad': 1,
        'cutoff_12th': 91.0,
        'entrance_score': 160.0,
        'college_tier': 'Tier-1 (Autonomous)',
        'tuition': 180000,
        'distance_km': 8.0,
        'competing_offers': 3,
        'merit_aid_pct': 0.10,
        'need_aid_pct': 0.05,
        'total_aid_pct': 0.08,
        'aid_amount': 14400,
        'net_price': 165600
    },

    {
        'district': 'Villupuram',
        'category': 'SC',
        'urban': 0,
        'family_income': 95000,
        'first_gen': 1,
        'parent_grad': 0,
        'cutoff_12th': 71.0,
        'entrance_score': 100.0,
        'college_tier': 'Tier-3 (Self-financing)',
        'tuition': 85000,
        'distance_km': 70.0,
        'competing_offers': 0,
        'merit_aid_pct': 0.15,
        'need_aid_pct': 0.55,
        'total_aid_pct': 0.35,
        'aid_amount': 29800,
        'net_price': 55200
    },

    {
        'district': 'Tiruppur',
        'category': 'BC',
        'urban': 1,
        'family_income': 340000,
        'first_gen': 0,
        'parent_grad': 1,
        'cutoff_12th': 84.0,
        'entrance_score': 138.0,
        'college_tier': 'Tier-2 (Affiliated)',
        'tuition': 110000,
        'distance_km': 22.0,
        'competing_offers': 2,
        'merit_aid_pct': 0.25,
        'need_aid_pct': 0.15,
        'total_aid_pct': 0.20,
        'aid_amount': 22000,
        'net_price': 88000
    }
]

# Convert the applicant list into a Pandas DataFrame.
batch_df = pd.DataFrame(applicant_batch)

# Generate the predicted enrollment probability for every applicant.
batch_df['predicted_yield_prob'] = model.predict_proba(
    batch_df[num_feats + cat_feats]
)[:, 1]

# Display the important fields and predicted yield probability.
batch_df[
    [
        'district',
        'category',
        'total_aid_pct',
        'net_price',
        'predicted_yield_prob'
    ]
]

,district,category,total_aid_pct,net_price,predicted_yield_prob
0,Chennai,OC,0.08,165600,0.691935
1,Villupuram,SC,0.35,55200,0.257133
2,Tiruppur,BC,0.20,88000,0.653328


In [22]:
# Calculate a normalized financial-need index from family income.
def compute_need_index(
    family_income,
    income_ceiling=4_500_000
):

    # Convert family income into a need score.
    # Lower income produces a higher need index.
    return float(
        np.clip(
            1 - (family_income / income_ceiling),
            0.02,
            0.98
        )
    )


# Search different aid percentages and identify
# the package with the highest expected revenue.
def optimize_aid_package(
    applicant: dict,
    model,
    num_feats,
    cat_feats,
    aid_pct_grid=np.arange(0.0, 0.81, 0.05),
    min_yield_prob=0.45,
    equity_floor_pct=0.25,
    need_floor_threshold=0.6,
    income_ceiling=4_500_000
):

    # Calculate the applicant's financial-need index.
    need_idx = compute_need_index(
        applicant['family_income'],
        income_ceiling
    )

    # Create the initial list of aid percentages to evaluate.
    grid = aid_pct_grid.copy()

    # Apply the minimum aid requirement for high-need applicants.
    if need_idx >= need_floor_threshold:

        # Keep only aid percentages meeting the equity floor.
        grid = (
            grid[grid >= equity_floor_pct]
            if (grid >= equity_floor_pct).any()
            else grid
        )

    # Create an empty list for candidate aid packages.
    candidates = []

    # Evaluate every aid percentage in the grid.
    for pct in grid:

        # Create a copy of the applicant information.
        row = dict(applicant)

        # Set the proposed total aid percentage.
        row['total_aid_pct'] = pct

        # Calculate the monetary aid amount.
        row['aid_amount'] = round(
            row['tuition'] * pct,
            -2
        )

        # Calculate the resulting net price.
        row['net_price'] = (
            row['tuition'] - row['aid_amount']
        )

        # Store the candidate package.
        candidates.append(row)

    # Convert all candidate packages into a DataFrame.
    cand_df = pd.DataFrame(candidates)

    # Predict enrollment probability for every candidate package.
    proba = model.predict_proba(
        cand_df[num_feats + cat_feats]
    )[:, 1]

    # Store the predicted enrollment probabilities.
    cand_df['yield_prob'] = proba

    # Calculate expected revenue:
    # predicted enrollment probability × net price.
    cand_df['expected_revenue'] = (
        cand_df['yield_prob'] *
        cand_df['net_price']
    )

    # Keep packages that satisfy the minimum yield probability.
    feasible = cand_df[
        cand_df['yield_prob'] >= min_yield_prob
    ]

    # If no package meets the yield requirement,
    # use all candidate packages instead.
    pool = feasible if len(feasible) else cand_df

    # Select the package with the highest expected revenue.
    best = pool.loc[
        pool['expected_revenue'].idxmax()
    ].to_dict()

    # Store the applicant's calculated need index.
    best['need_index'] = need_idx

    # Record whether the yield-probability floor was satisfied.
    best['met_yield_floor'] = len(feasible) > 0

    # Return the selected package.
    return best

In [23]:
# Generate the optimized aid package for the sample applicant.
best_package = optimize_aid_package(
    sample_applicant,
    model,
    num_feats,
    cat_feats
)

# Display the optimized aid-package details.
print('Recommended aid package for the Section 4 applicant:')

# Display the important optimization results.
for k in [
    'total_aid_pct',
    'aid_amount',
    'net_price',
    'yield_prob',
    'expected_revenue',
    'need_index',
    'met_yield_floor'
]:
    print(f"  {k}: {best_package[k]}")

Recommended aid package for the Section 4 applicant:
  total_aid_pct: 0.55
  aid_amount: 60500.0
  net_price: 49500.0
  yield_prob: 0.515819693345024
  expected_revenue: 25533.074820578688
  need_index: 0.9533333333333334
  met_yield_floor: True
